# AstroML long-window snapshot memory experiment

This notebook measures memory usage and validates the chunked snapshot path for long windows.

In [1]:
import gc
import tracemalloc
import time

try:
    import psutil
except Exception as exc:
    psutil = None
    print('psutil unavailable', exc)


def measure_memory(label='snapshot'):
    gc.collect()
    tracemalloc.start()
    start = time.perf_counter()
    rss_mb = psutil.Process().memory_info().rss / (1024 * 1024) if psutil else None
    current, peak = tracemalloc.get_traced_memory()
    elapsed = time.perf_counter() - start
    print(f'[{label}] elapsed={elapsed:.3f}s current_mb={current / 1024 / 1024:.2f} peak_mb={peak / 1024 / 1024:.2f}')
    if rss_mb is not None:
        print(f'[{label}] rss_mb={rss_mb:.2f}')

measure_memory('baseline')

[baseline] elapsed=0.000s current_mb=0.00 peak_mb=0.04
[baseline] rss_mb=69.65


In [ ]:
from astroml.features.graph.snapshot import Edge, iter_db_snapshots

print('iter_db_snapshots now accepts chunk_size to keep DB fetches bounded.')

In [ ]:
def build_snapshot_in_chunks(edges, chunk_size=1000):
    chunk = []
    for edge in edges:
        chunk.append(edge)
        if len(chunk) >= chunk_size:
            yield chunk
            chunk = []
    if chunk:
        yield chunk

print('Chunked builder ready; use chunk_size to keep memory bounded.')

In [ ]:
demo_edges = [Edge(src=f'u{i%4}', dst=f'v{i%3}', timestamp=1_700_000_000 + i * 60) for i in range(12)]
chunks = list(build_snapshot_in_chunks(demo_edges, chunk_size=5))
assert sum(len(chunk) for chunk in chunks) == len(demo_edges)
print('chunk_count=', len(chunks), 'total_edges=', sum(len(chunk) for chunk in chunks))

In [ ]:
# Optional: compare the chunked path with the baseline on a representative long-window input.
# Replace this with your actual ledger snapshot builder when you run the notebook on real data.

In [5]:
import pytest
pytest.main(['-q', 'tests/test_snapshot_memory.py'])

.                                                                        [100%]
1 passed in 0.09s


<ExitCode.OK: 0>